# Frozen DNA language-model embeddings on a free Colab GPU

This notebook runs the only step that needs a GPU: extracting frozen embeddings from a plant DNA
language model for the Plants Genomic Benchmark gene-expression task. Everything else (k-mer
baseline, ridge head, figures) runs on your laptop.

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

Settings below must match the ones you use for the k-mer baseline, so both see the same training subset.

In [ ]:
SPECIES   = "arabidopsis_thaliana"   # smallest task: about 25.7k train / 3.4k valid / 3.4k test
MAX_TRAIN = 5000                     # same subset as `make baseline MAX_TRAIN=5000`
MODEL     = "plantcaduceus-l20"      # or plantcaduceus-l24 / plantcaduceus-l32 / agront-1b

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/Olaraym/plant-dnalm-expression.git
%cd plant-dnalm-expression

## Install

`mamba-ssm` and `causal-conv1d` are needed by PlantCaduceus and can take 10 to 20 minutes to build.
If the install fails, switch `MODEL` to `"agront-1b"` (no extra packages; slower, so use `MAX_TRAIN = 2000`).

In [ ]:
!pip install -q transformers huggingface_hub tqdm
if MODEL.startswith("plantcaduceus"):
    !pip install -q causal-conv1d mamba-ssm --no-build-isolation

## Download the task and confirm the label layout

In [ ]:
!python scripts/download_data.py --species $SPECIES

## Extract embeddings

Each 6 kb sequence is cut into windows the model can read (512 bp for PlantCaduceus); the last hidden
layer is averaged over tokens and windows. The model is frozen: no weights are trained here.

In [ ]:
!python scripts/extract_embeddings.py --species $SPECIES --model $MODEL --max-train $MAX_TRAIN --batch-size 32

## Fit the ridge head and compare with the k-mer baseline (CPU is fine)

In [ ]:
!python scripts/run_embedding_ridge.py --species $SPECIES --model $MODEL --max-train $MAX_TRAIN
!python scripts/run_kmer_baseline.py --species $SPECIES --max-train $MAX_TRAIN --k 6
!python scripts/compare.py

In [ ]:
from IPython.display import Image
Image("results/figures/main.png")

## Save the outputs

Download the zip and unzip it into your local clone, then commit `results/` (the embeddings are large
and git-ignored; keep them on Google Drive if you want to reuse them).

In [ ]:
!zip -qr outputs.zip results data/embeddings
from google.colab import files
files.download("outputs.zip")